In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, special
from ipywidgets import RadioButtons, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# LOW-PASS PROTOTYPE TO BAND-STOP FREQUENCY TRANSFORMATION
#
# Exercise specifications:
#
#       ωp1 = 20 rad/s
#       ωp2 = 80 rad/s
#       ωs1 = 48 rad/s
#       ωs2 = 52 rad/s
#       Ap  = 1.0 dB
#       As  = 25.0 dB
#
# Prototype:
#
#       Elliptic / Cauer
#
# Frequency transformation:
#
#                    p B
#       s = ----------------------
#               p² + ω0²
#
# where:
#
#       ω0 = sqrt(ωp1 ωp2)
#
# Frequency mapping:
#
#                     B ω
#       Ω = ----------------------
#                 ω0² - ω²
#
# Prototype-pole mapping:
#
#                     B
#       p² - ---------------- p + ω0² = 0
#                    sk
#
# Therefore every prototype pole produces TWO band-stop poles.
#
# Prototype finite-zero mapping:
#
#                     B
#       p² - ---------------- p + ω0² = 0
#                    zk
#
# Therefore every finite prototype zero produces TWO transformed zeros.
#
# Each prototype zero at infinity produces a pair of zeros:
#
#       p = ±jω0
#
# Hence:
#
#       prototype order = N
#       band-stop order  = 2N
#
# ==============================================================================
# DISPLAY STRATEGY
# ==============================================================================
#
# ONLY ONE FILTER IS DISPLAYED AT A TIME.
#
# Filter View:
#
#       Prototype LP
#       Transformed BS
#
# Displayed Quantity:
#
#       Pole-Zero Diagram
#       Magnitude Response
#       Phase Response
#       Group Delay
#       Impulse Response
#       Step Response
#
# The numerical calculation area is split into TWO COLUMNS:
#
#       Column 1 -> Steps 1-6
#       Column 2 -> Steps 7-12
#
# DISPLAY MODIFICATION:
#
# The canvas dimensions remain exactly:
#
#       850 x 540 pixels
#
# The plotting rectangle is moved slightly toward the LEFT inside the canvas.
# Its width is NOT increased.
#
# Original width:
#
#       0.76 - 0.12 = 0.64
#
# New width:
#
#       0.715 - 0.075 = 0.64
#
# Therefore only the internal position of the graph changes.
#
# Legends are displayed BELOW the graph with their entries arranged
# horizontally on one line.
#
# ==============================================================================
# NUMERICAL-STABILITY MODIFICATION
# ==============================================================================
#
# The previous version constructed scipy.signal.TransferFunction objects from
# high-order numerator and denominator coefficient arrays.
#
# This may generate:
#
#       BadCoefficients:
#       Badly conditioned filter coefficients (numerator)
#
# The filter itself is valid. The warning is caused by the poor numerical
# conditioning of the polynomial representation.
#
# In this version the time-domain systems are constructed DIRECTLY from
# zeros, poles and gain using:
#
#       signal.zpk2ss(...)
#
# followed by:
#
#       signal.StateSpace(...)
#
# This avoids unnecessary normalization of badly scaled numerator
# coefficients and therefore eliminates the BadCoefficients warning at its
# source rather than merely suppressing it.
#
# The transformed band-stop system contains a direct-feedthrough term.
# scipy.signal.impulse applied to the state-space system returns the regular
# finite part of the impulse response. The Dirac impulse at t = 0 is shown
# separately with a marker.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# EXERCISE SPECIFICATIONS
# ==============================================================================

wp1 = 20.0
wp2 = 80.0
ws1_original = 48.0
ws2_original = 52.0
Ap = 1.0
As = 25.0

# ==============================================================================
# STEP 1: BASIC ATTENUATION PARAMETER
# ==============================================================================

D = (10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0)

k1 = 1.0 / np.sqrt(D)

k1_complement = np.sqrt(1.0 - k1**2)

# ==============================================================================
# STEP 2: INITIAL GEOMETRIC CENTER AND PASSBAND WIDTH
# ==============================================================================

omega0 = np.sqrt(wp1 * wp2)

Bp = wp2 - wp1

# ==============================================================================
# STEP 3: STOPBAND FREQUENCY READJUSTMENT
# ==============================================================================

ws2_redefined = wp1 * wp2 / ws1_original

Omega_s_candidate_1 = abs(ws2_redefined - ws1_original) / Bp

ws1_redefined = wp1 * wp2 / ws2_original

Omega_s_candidate_2 = abs(ws2_original - ws1_redefined) / Bp

k2 = max(Omega_s_candidate_1, Omega_s_candidate_2)

k2_complement = np.sqrt(1.0 - k2**2)

# ==============================================================================
# STEP 4: ELLIPTIC INTEGRALS AND MINIMUM ORDER
#
#       N = (K1'/K1)(K2/K2')
#
# scipy.special.ellipk accepts m = k².
# ==============================================================================

K1 = special.ellipk(k1**2)

K1_complement = special.ellipk(k1_complement**2)

K2 = special.ellipk(k2**2)

K2_complement = special.ellipk(k2_complement**2)

N_exact = (K1_complement / K1) * (K2 / K2_complement)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 5: NORMALIZED PROTOTYPE FREQUENCIES
# ==============================================================================

Omega_p = np.sqrt(k2)

Omega_s = 1.0 / np.sqrt(k2)

# ==============================================================================
# STEP 6: BAND-STOP BANDWIDTH PARAMETER
# ==============================================================================

B = Omega_p * (wp2 - wp1)

# ==============================================================================
# STEP 7: EXACT ELLIPTIC / CAUER PROTOTYPE
#
# The order is calculated above from the theory.
#
# scipy.signal.ellip is used to evaluate the corresponding poles,
# finite zeros and gain numerically.
# ==============================================================================

prototype_zeros, prototype_poles, prototype_gain = signal.ellip(N, Ap, As, Omega_p, btype='low', analog=True, output='zpk')

prototype_zeros = np.asarray(prototype_zeros, dtype=complex)

prototype_poles = np.asarray(prototype_poles, dtype=complex)

prototype_gain = float(np.real_if_close(prototype_gain))

num_prototype_zeros_at_infinity = N - len(prototype_zeros)

# ==============================================================================
# STEP 8: PROTOTYPE TRANSFER FUNCTION
#
# Polynomial coefficients are retained for:
#
#       frequency response
#       printed transfer function
#       specification verification
#
# They are NOT used to construct the time-domain scipy system.
# ==============================================================================

prototype_num, prototype_den = signal.zpk2tf(prototype_zeros, prototype_poles, prototype_gain)

prototype_num = np.real_if_close(prototype_num, tol=1000).real

prototype_den = np.real_if_close(prototype_den, tol=1000).real

# ==============================================================================
# STEP 9: BAND-STOP POLE MAPPING
# ==============================================================================

transformed_poles_list = []

for prototype_pole in prototype_poles:

    pole_linear_coefficient = B / prototype_pole

    discriminant = complex(pole_linear_coefficient**2 - 4.0 * omega0**2)

    root_discriminant = np.sqrt(discriminant)

    transformed_poles_list.append((pole_linear_coefficient + root_discriminant) / 2.0)

    transformed_poles_list.append((pole_linear_coefficient - root_discriminant) / 2.0)

transformed_poles = np.asarray(transformed_poles_list, dtype=complex)

# ==============================================================================
# STEP 10: BAND-STOP ZERO MAPPING
# ==============================================================================

transformed_zeros_list = []

for prototype_zero in prototype_zeros:

    zero_linear_coefficient = B / prototype_zero

    discriminant = complex(zero_linear_coefficient**2 - 4.0 * omega0**2)

    root_discriminant = np.sqrt(discriminant)

    transformed_zeros_list.append((zero_linear_coefficient + root_discriminant) / 2.0)

    transformed_zeros_list.append((zero_linear_coefficient - root_discriminant) / 2.0)

for index in range(num_prototype_zeros_at_infinity):

    transformed_zeros_list.append(1j * omega0)

    transformed_zeros_list.append(-1j * omega0)

transformed_zeros = np.asarray(transformed_zeros_list, dtype=complex)

# ==============================================================================
# STEP 11: BAND-STOP GAIN
#
# Since:
#
#       HBS(0) = HLPP(0)
#
# the transformed gain follows from the DC-gain equality.
# ==============================================================================

prototype_dc_gain = prototype_gain * np.prod(-prototype_zeros) / np.prod(-prototype_poles)

prototype_dc_gain = np.real_if_close(prototype_dc_gain, tol=1000).real

transformed_gain = prototype_dc_gain * np.prod(-transformed_poles) / np.prod(-transformed_zeros)

transformed_gain = np.real_if_close(transformed_gain, tol=1000).real

# ==============================================================================
# STEP 12: BAND-STOP TRANSFER FUNCTION
# ==============================================================================

transformed_num, transformed_den = signal.zpk2tf(transformed_zeros, transformed_poles, transformed_gain)

transformed_num = np.real_if_close(transformed_num, tol=1000).real

transformed_den = np.real_if_close(transformed_den, tol=1000).real

# ==============================================================================
# FREQUENCY AXES
# ==============================================================================

Omega_axis = np.logspace(-3, 2, 6000)

omega_axis = np.logspace(-1, 3, 7000)

# ==============================================================================
# PROTOTYPE FREQUENCY RESPONSE
# ==============================================================================

_, H_prototype = signal.freqs(prototype_num, prototype_den, worN=Omega_axis)

prototype_magnitude = np.abs(H_prototype)

prototype_phase = np.unwrap(np.angle(H_prototype))

prototype_phase_deg = np.rad2deg(prototype_phase)

prototype_group_delay = -np.gradient(prototype_phase, Omega_axis)

# ==============================================================================
# TRANSFORMED BAND-STOP FREQUENCY RESPONSE
# ==============================================================================

_, H_transformed = signal.freqs(transformed_num, transformed_den, worN=omega_axis)

transformed_magnitude = np.abs(H_transformed)

transformed_phase = np.unwrap(np.angle(H_transformed))

transformed_phase_deg = np.rad2deg(transformed_phase)

transformed_group_delay = -np.gradient(transformed_phase, omega_axis)

# ==============================================================================
# TIME-DOMAIN SYSTEMS
#
# IMPORTANT NUMERICAL CHANGE:
#
# Do NOT use:
#
#       signal.TransferFunction(num, den)
#
# Instead construct state-space realizations directly from zeros, poles
# and gain. This avoids the BadCoefficients warning.
# ==============================================================================

A_prototype, B_prototype, C_prototype, D_prototype = signal.zpk2ss(prototype_zeros, prototype_poles, prototype_gain)

prototype_system = signal.StateSpace(A_prototype, B_prototype, C_prototype, D_prototype)

A_transformed, B_transformed, C_transformed, D_transformed = signal.zpk2ss(transformed_zeros, transformed_poles, transformed_gain)

transformed_system = signal.StateSpace(A_transformed, B_transformed, C_transformed, D_transformed)

# ==============================================================================
# PROTOTYPE TIME SCALE
# ==============================================================================

prototype_slowest_rate = np.min(np.abs(np.real(prototype_poles)))

prototype_time_constant = 1.0 / prototype_slowest_rate

t_prototype_max = 12.0 * prototype_time_constant

t_prototype = np.linspace(0.0, t_prototype_max, 6000)

# ==============================================================================
# TRANSFORMED TIME SCALE
# ==============================================================================

transformed_slowest_rate = np.min(np.abs(np.real(transformed_poles)))

transformed_time_constant = 1.0 / transformed_slowest_rate

t_transformed_max = 12.0 * transformed_time_constant

t_transformed = np.linspace(0.0, t_transformed_max, 7000)

# ==============================================================================
# PROTOTYPE TIME RESPONSES
# ==============================================================================

t_impulse_prototype, h_prototype = signal.impulse(prototype_system, T=t_prototype)

t_step_prototype, step_prototype = signal.step(prototype_system, T=t_prototype)

# ==============================================================================
# BAND-STOP IMPULSE RESPONSE
#
# The state-space matrix D contains the direct-feedthrough coefficient.
#
# Therefore:
#
#       hBS(t) = D δ(t) + regular finite response
#
# scipy.signal.impulse returns the finite regular part. The Dirac term is
# represented separately by a marker.
# ==============================================================================

direct_gain = float(np.asarray(D_transformed).squeeze())

t_impulse_transformed, h_transformed_regular = signal.impulse(transformed_system, T=t_transformed)

# ==============================================================================
# BAND-STOP STEP RESPONSE
# ==============================================================================

t_step_transformed, step_transformed = signal.step(transformed_system, T=t_transformed)

# ==============================================================================
# FORCE ONE-DIMENSIONAL RESPONSE ARRAYS
# ==============================================================================

h_prototype = np.asarray(h_prototype).squeeze()

step_prototype = np.asarray(step_prototype).squeeze()

h_transformed_regular = np.asarray(h_transformed_regular).squeeze()

step_transformed = np.asarray(step_transformed).squeeze()

# ==============================================================================
# ATTENUATION CALCULATION
# ==============================================================================

def attenuation_from_transfer_function(b, a, frequency):

    _, H = signal.freqs(b, a, worN=np.asarray([frequency]))

    magnitude = max(np.abs(H[0]), 1e-15)

    return -20.0 * np.log10(magnitude)

# ==============================================================================
# SPECIFICATION VERIFICATION
# ==============================================================================

Ap1_actual = attenuation_from_transfer_function(transformed_num, transformed_den, wp1)

Ap2_actual = attenuation_from_transfer_function(transformed_num, transformed_den, wp2)

As1_actual = attenuation_from_transfer_function(transformed_num, transformed_den, ws1_original)

As2_actual = attenuation_from_transfer_function(transformed_num, transformed_den, ws2_original)

# ==============================================================================
# AUXILIARY MAPPED FREQUENCIES
# ==============================================================================

Omega_at_wp1 = abs(B * wp1 / (omega0**2 - wp1**2))

Omega_at_wp2 = abs(B * wp2 / (omega0**2 - wp2**2))

Omega_at_ws1 = abs(B * ws1_original / (omega0**2 - ws1_original**2))

Omega_at_ws2 = abs(B * ws2_original / (omega0**2 - ws2_original**2))

# ==============================================================================
# POLYNOMIAL STRING
# ==============================================================================

def polynomial_string(coefficients, variable='p'):

    degree = len(coefficients) - 1

    terms = []

    for index, coefficient in enumerate(coefficients):

        power = degree - index

        if abs(coefficient) < 1e-8:

            continue

        if power == 0:

            terms.append(f'{coefficient:.6e}')

        elif power == 1:

            terms.append(f'{coefficient:.6e}{variable}')

        else:

            terms.append(f'{coefficient:.6e}{variable}^{power}')

    return ' + '.join(terms)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1580px;
    max-width:1580px;
    box-sizing:border-box;
">

<b>Low-Pass Prototype to Band-Stop Frequency Transformation</b><br>

An elliptic/Cauer prototype is constructed for a band-stop filter with
ω<sub>p1</sub> = {wp1:.0f} rad/s,
ω<sub>p2</sub> = {wp2:.0f} rad/s,
ω<sub>s1</sub> = {ws1_original:.0f} rad/s,
ω<sub>s2</sub> = {ws2_original:.0f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.1f} dB.

The transformation is

<b>s = pB/(p² + ω<sub>0</sub>²)</b>.

<br>

<b>Visualization strategy:</b>
The normalized prototype and the transformed physical band-stop filter are
displayed separately because their frequency, pole and time scales differ
substantially. Use the <b>Filter View</b> selector to inspect either filter
with axis limits appropriate to its own scale.

</div>
""", layout=Layout(width='1590px', max_width='1590px'))

# ==============================================================================
# FILTER-VIEW SELECTOR
# ==============================================================================

filter_view_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Filter View
</div>
""")

filter_view_selector = RadioButtons(options=['Prototype LP', 'Transformed BS'], value='Prototype LP', description='', layout=Layout(width='175px'))

filter_view_panel = VBox([filter_view_title, filter_view_selector], layout=Layout(width='210px', min_width='210px', max_width='210px', border='1px solid #cccccc', padding='9px', align_items='flex-start'))

# ==============================================================================
# DISPLAY SELECTOR
# ==============================================================================

display_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Displayed Quantity
</div>
""")

display_selector = RadioButtons(options=['Pole-Zero Diagram', 'Magnitude Response', 'Phase Response', 'Group Delay', 'Impulse Response', 'Step Response'], value='Pole-Zero Diagram', description='', layout=Layout(width='185px'))

display_panel = VBox([display_title, display_selector], layout=Layout(width='210px', min_width='210px', max_width='210px', border='1px solid #cccccc', padding='9px', align_items='flex-start'))

# ==============================================================================
# INFORMATION PANELS
# ==============================================================================

info_left = HTML(layout=Layout(width='315px', max_width='315px'))

info_right = HTML(layout=Layout(width='340px', max_width='340px'))

# ==============================================================================
# TEXT REPRESENTATIONS
# ==============================================================================

prototype_pole_text = '<br>'.join([f's{k + 1} = {pole.real:+.6f} {pole.imag:+.6f}j' for k, pole in enumerate(prototype_poles)])

prototype_zero_text = '<br>'.join([f'z{k + 1} = {zero.real:+.6f} {zero.imag:+.6f}j' for k, zero in enumerate(prototype_zeros)])

transformed_pole_text = '<br>'.join([f'p{k + 1} = {pole.real:+.6f} {pole.imag:+.6f}j' for k, pole in enumerate(transformed_poles)])

transformed_zero_text = '<br>'.join([f'z{k + 1} = {zero.real:+.6f} {zero.imag:+.6f}j' for k, zero in enumerate(transformed_zeros)])

numerator_text = polynomial_string(transformed_num)

denominator_text = polynomial_string(transformed_den)

Ap1_status = '✓ satisfied' if Ap1_actual <= Ap + 1e-8 else '✗ not satisfied'

Ap2_status = '✓ satisfied' if Ap2_actual <= Ap + 1e-8 else '✗ not satisfied'

As1_status = '✓ satisfied' if As1_actual >= As - 1e-8 else '✗ not satisfied'

As2_status = '✓ satisfied' if As2_actual >= As - 1e-8 else '✗ not satisfied'

# ==============================================================================
# LEFT INFORMATION COLUMN
# ==============================================================================

info_left.value = f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:9px 10px;
    font-size:11.1px;
    line-height:1.48;
    background:white;
    width:310px;
    box-sizing:border-box;
">

<b>Step 1 — Specifications</b><br>

<span style="color:#0066cc;">
ωp1 = {wp1:.0f}, ωp2 = {wp2:.0f} rad/s<br>
ωs1 = {ws1_original:.0f}, ωs2 = {ws2_original:.0f} rad/s<br>
Ap = {Ap:.1f} dB, As = {As:.1f} dB
</span>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 2 — Attenuation parameter</b><br>

D =
<span style="color:#0066cc;">{D:.6f}</span><br>

k₁ =
<span style="color:#0066cc;">{k1:.6f}</span><br>

k₁′ =
<span style="color:#0066cc;">{k1_complement:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 3 — Frequency readjustment</b><br>

ω₀ = √(ωp1ωp2) =
<span style="color:#0066cc;">{omega0:.6f}</span><br>

Bp =
<span style="color:#0066cc;">{Bp:.6f}</span><br>

ωs2′ =
<span style="color:#0066cc;">{ws2_redefined:.6f}</span><br>

Candidate 1 =
<span style="color:#0066cc;">{Omega_s_candidate_1:.6f}</span><br>

ωs1′ =
<span style="color:#0066cc;">{ws1_redefined:.6f}</span><br>

Candidate 2 =
<span style="color:#0066cc;">{Omega_s_candidate_2:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 4 — Elliptic selectivity parameter</b><br>

k₂ =
<span style="color:#0066cc;"><b>{k2:.6f}</b></span><br>

k₂′ =
<span style="color:#0066cc;">{k2_complement:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 5 — Minimum elliptic order</b><br>

K₁ =
<span style="color:#0066cc;">{K1:.6f}</span><br>

K₁′ =
<span style="color:#0066cc;">{K1_complement:.6f}</span><br>

K₂ =
<span style="color:#0066cc;">{K2:.6f}</span><br>

K₂′ =
<span style="color:#0066cc;">{K2_complement:.6f}</span><br>

Nmin =
<span style="color:#0066cc;">{N_exact:.6f}</span><br>

N =
<span style="color:#0066cc;"><b>{N}</b></span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 6 — Normalized frequencies and transformation</b><br>

Ωp =
<span style="color:#0066cc;">{Omega_p:.6f}</span><br>

Ωs =
<span style="color:#0066cc;">{Omega_s:.6f}</span><br>

ω₀ =
<span style="color:#0066cc;">{omega0:.6f} rad/s</span><br>

B =
<span style="color:#0066cc;"><b>{B:.6f} rad/s</b></span>

</div>

</div>
"""

# ==============================================================================
# RIGHT INFORMATION COLUMN
# ==============================================================================

info_right.value = f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:9px 10px;
    font-size:11.1px;
    line-height:1.48;
    background:white;
    width:335px;
    box-sizing:border-box;
">

<b>Step 7 — Prototype poles</b><br>

<span style="color:#0066cc;">
{prototype_pole_text}
</span>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 8 — Prototype zeros and gain</b><br>

<span style="color:#0066cc;">
{prototype_zero_text}
</span><br>

Zeros at infinity =
<span style="color:#0066cc;">
{num_prototype_zeros_at_infinity}
</span><br>

Prototype gain =
<span style="color:#0066cc;">
{prototype_gain:.9f}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 9 — Transformed poles</b><br>

<span style="color:#0066cc;">
{transformed_pole_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 10 — Transformed zeros</b><br>

<span style="color:#0066cc;">
{transformed_zero_text}
</span><br>

Band-stop order =
<span style="color:#0066cc;">
<b>{2 * N}</b>
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 11 — Final transfer function</b><br>

H<sub>BS</sub>(p) = N(p) / D(p)<br><br>

<span style="color:#0066cc;">
N(p) = {numerator_text}<br><br>
D(p) = {denominator_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 12 — Specification verification</b><br>

A(ωp1) =
<span style="color:#0066cc;">
{Ap1_actual:.6f} dB
</span>
→ {Ap1_status}<br>

A(ωp2) =
<span style="color:#0066cc;">
{Ap2_actual:.6f} dB
</span>
→ {Ap2_status}<br>

A(ωs1) =
<span style="color:#0066cc;">
{As1_actual:.6f} dB
</span>
→ {As1_status}<br>

A(ωs2) =
<span style="color:#0066cc;">
{As2_actual:.6f} dB
</span>
→ {As2_status}<br><br>

Mapped Ω at ωp1 =
<span style="color:#0066cc;">
{Omega_at_wp1:.6f}
</span><br>

Mapped Ω at ωp2 =
<span style="color:#0066cc;">
{Omega_at_wp2:.6f}
</span><br>

Mapped Ω at ωs1 =
<span style="color:#0066cc;">
{Omega_at_ws1:.6f}
</span><br>

Mapped Ω at ωs2 =
<span style="color:#0066cc;">
{Omega_at_ws2:.6f}
</span>

</div>

</div>
"""

# ==============================================================================
# MAIN FIGURE
#
# Canvas:
#
#       850 x 540 pixels
#
# Plot width is unchanged.
#
# Old:
#
#       left = 0.12
#       right = 0.76
#
# New:
#
#       left = 0.075
#       right = 0.715
#
# Both have width 0.64.
#
# Extra bottom space is reserved for the horizontal legend.
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.5, 5.5))

response_line, = ax.plot([], [], linewidth=2.3, color='red')

pole_line, = ax.plot([], [], 'ro', markersize=7)

zero_line, = ax.plot([], [], 'gx', markersize=9, markeredgewidth=1.8)

horizontal_axis = ax.axhline(0.0, color='black', linewidth=0.8)

vertical_axis = ax.axvline(0.0, color='black', linewidth=0.8)

passband_line_1 = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

passband_line_2 = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

stopband_line_1 = ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.0)

stopband_line_2 = ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.0)

impulse_marker, = ax.plot([], [], 'r^', markersize=9)

ax.grid(True, linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

# ==============================================================================
# CRITICAL DISPLAY MODIFICATION
# ==============================================================================

fig.subplots_adjust(left=0.075, right=0.715, bottom=0.27, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '850px'

fig.canvas.layout.height = '540px'

# ==============================================================================
# HORIZONTAL LEGEND BELOW GRAPH
# ==============================================================================

def external_legend(handles, labels):

    ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.18), borderaxespad=0.0, fontsize=8, frameon=True, title='Legend', title_fontsize=9, labelspacing=1.0, handlelength=2.5, ncol=len(labels), columnspacing=2.2)

# ==============================================================================
# MAIN UPDATE FUNCTION
# ==============================================================================

def update_plot(change=None):

    view = filter_view_selector.value

    selected = display_selector.value

    # --------------------------------------------------------------------------
    # RESET VISIBILITY
    # --------------------------------------------------------------------------

    response_line.set_visible(False)

    pole_line.set_visible(False)

    zero_line.set_visible(False)

    horizontal_axis.set_visible(False)

    vertical_axis.set_visible(False)

    passband_line_1.set_visible(False)

    passband_line_2.set_visible(False)

    stopband_line_1.set_visible(False)

    stopband_line_2.set_visible(False)

    impulse_marker.set_visible(False)

    old_legend = ax.get_legend()

    if old_legend is not None:

        old_legend.remove()

    # ==========================================================================
    # PROTOTYPE ELLIPTIC LOW-PASS FILTER
    # ==========================================================================

    if view == 'Prototype LP':

        response_line.set_color('blue')

        pole_line.set_color('blue')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            zero_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(prototype_poles), np.imag(prototype_poles))

            zero_line.set_data(np.real(prototype_zeros), np.imag(prototype_zeros))

            all_values = np.concatenate([prototype_poles, prototype_zeros])

            real_limit = max(np.max(np.abs(np.real(all_values))), 1.0)

            imag_limit = max(np.max(np.abs(np.imag(all_values))), 1.0)

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-1.25 * real_limit, 0.25 * real_limit)

            ax.set_ylim(-1.20 * imag_limit, 1.20 * imag_limit)

            ax.set_aspect('auto')

            ax.set_xlabel('Re{s}', fontsize=10)

            ax.set_ylabel('Im{s}', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Pole-Zero Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line, zero_line], ['Prototype poles', 'Finite prototype zeros'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line_1.set_visible(True)

            stopband_line_1.set_visible(True)

            response_line.set_data(Omega_axis, prototype_magnitude)

            passband_line_1.set_xdata([Omega_p, Omega_p])

            stopband_line_1.set_xdata([Omega_s, Omega_s])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('|H_LPP(jΩ)|', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(Omega_axis, prototype_phase_deg)

            phase_min = np.min(prototype_phase_deg)

            phase_max = np.max(prototype_phase_deg)

            phase_margin = 0.06 * max(phase_max - phase_min, 90.0)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(Omega_axis, prototype_group_delay)

            finite_gd = prototype_group_delay[np.isfinite(prototype_group_delay)]

            finite_gd = finite_gd[finite_gd >= 0.0]

            gd_max = max(np.max(finite_gd), 1e-6)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(1e-3, 100.0)

            ax.set_ylim(0.0, 1.10 * gd_max)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_prototype, h_prototype)

            y_min = np.min(h_prototype)

            y_max = np.max(h_prototype)

            y_margin = 0.10 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_prototype_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('h_LPP(t)', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Impulse Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_prototype, step_prototype)

            y_min = min(0.0, np.min(step_prototype))

            y_max = max(1.0, np.max(step_prototype))

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_prototype_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Prototype Elliptic/Cauer Step Response', fontsize=13, fontweight='bold', pad=8)

    # ==========================================================================
    # TRANSFORMED BAND-STOP FILTER
    # ==========================================================================

    elif view == 'Transformed BS':

        response_line.set_color('red')

        pole_line.set_color('red')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            zero_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(transformed_poles), np.imag(transformed_poles))

            zero_line.set_data(np.real(transformed_zeros), np.imag(transformed_zeros))

            all_values = np.concatenate([transformed_poles, transformed_zeros])

            real_limit = max(np.max(np.abs(np.real(all_values))), 1.0)

            imag_limit = max(np.max(np.abs(np.imag(all_values))), 1.0)

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-1.30 * real_limit, 0.30 * real_limit)

            ax.set_ylim(-1.10 * imag_limit, 1.10 * imag_limit)

            ax.set_aspect('auto')

            ax.set_xlabel('Re{p}', fontsize=10)

            ax.set_ylabel('Im{p}', fontsize=10)

            ax.set_title('Transformed Band-Stop Pole-Zero Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line, zero_line], ['Band-stop poles', 'Band-stop zeros'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line_1.set_visible(True)

            passband_line_2.set_visible(True)

            stopband_line_1.set_visible(True)

            stopband_line_2.set_visible(True)

            response_line.set_data(omega_axis, transformed_magnitude)

            passband_line_1.set_xdata([wp1, wp1])

            passband_line_2.set_xdata([wp2, wp2])

            stopband_line_1.set_xdata([ws1_original, ws1_original])

            stopband_line_2.set_xdata([ws2_original, ws2_original])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(5.0, 200.0)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('|H_BS(jω)|', fontsize=10)

            ax.set_title('Transformed Band-Stop Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(omega_axis, transformed_phase_deg)

            visible_mask = (omega_axis >= 5.0) & (omega_axis <= 200.0)

            visible_phase = transformed_phase_deg[visible_mask]

            phase_min = np.min(visible_phase)

            phase_max = np.max(visible_phase)

            phase_margin = 0.06 * max(phase_max - phase_min, 90.0)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(5.0, 200.0)

            ax.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Transformed Band-Stop Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(omega_axis, transformed_group_delay)

            visible_mask = (omega_axis >= 5.0) & (omega_axis <= 200.0)

            finite_gd = transformed_group_delay[visible_mask]

            finite_gd = finite_gd[np.isfinite(finite_gd)]

            finite_gd = finite_gd[finite_gd >= 0.0]

            gd_max = max(np.max(finite_gd), 1e-8)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(5.0, 200.0)

            ax.set_ylim(0.0, 1.10 * gd_max)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay (s)', fontsize=10)

            ax.set_title('Transformed Band-Stop Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            impulse_marker.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_transformed, h_transformed_regular)

            y_min = np.min(h_transformed_regular)

            y_max = np.max(h_transformed_regular)

            y_abs = max(abs(y_min), abs(y_max), 1.0)

            impulse_marker.set_data([0.0], [0.85 * y_abs])

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_transformed_max)

            ax.set_ylim(-1.10 * y_abs, 1.10 * y_abs)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('h_BS(t), regular part', fontsize=10)

            ax.set_title('Transformed Band-Stop Impulse Response', fontsize=13, fontweight='bold', pad=8)

            external_legend([response_line, impulse_marker], ['Regular part', f'{direct_gain:.3f} δ(t) at t = 0'])

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_transformed, step_transformed)

            y_min = min(0.0, np.min(step_transformed))

            y_max = max(1.0, np.max(step_transformed))

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_transformed_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Transformed Band-Stop Step Response', fontsize=13, fontweight='bold', pad=8)

    # --------------------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------------------

    horizontal_axis.set_ydata([0.0, 0.0])

    # --------------------------------------------------------------------------
    # GRID
    # --------------------------------------------------------------------------

    ax.grid(True, which='both', linestyle=':', alpha=0.5)

    # --------------------------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

filter_view_selector.observe(update_plot, names='value')

display_selector.observe(update_plot, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# LEFT CONTROL COLUMN
# ==============================================================================

left_column = VBox([filter_view_panel, display_panel], layout=Layout(width='220px', min_width='220px', max_width='220px', align_items='flex-start'))

# ==============================================================================
# NUMERICAL COLUMNS
# ==============================================================================

information_left_column = VBox([info_left], layout=Layout(width='320px', min_width='320px', max_width='320px', align_items='flex-start'))

information_right_column = VBox([info_right], layout=Layout(width='345px', min_width='345px', max_width='345px', align_items='flex-start'))

# ==============================================================================
# PLOT COLUMN
#
# No negative margin is used.
#
# The canvas itself therefore remains completely outside the numerical frame.
# Only the Axes has been repositioned inside its own 850-pixel canvas.
# ==============================================================================

plot_column = VBox([fig.canvas], layout=Layout(width='850px', min_width='850px', max_width='850px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# MAIN LAYOUT
# ==============================================================================

main_layout = HBox([left_column, information_left_column, information_right_column, plot_column], layout=Layout(width='1740px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)